In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from functions.full_model import get_predict_fn
from functions.evaluation import evaluate
from networks.cnn_network import CNNModel
from networks.cnn_network import build_dataloaders

con = duckdb.connect('../capillary.db')
negatives = con.execute(""" 
                 SELECT id FROM protein_data WHERE array_contains(analysis,'0085') AND interpretation ILIKE '%ingen%m%komponent%påvisas%immunfixation%utförd%';
                 """).fetchnumpy()['id']
positives = con.execute(""" 
                 SELECT id FROM protein_data WHERE array_contains(analysis,'0085') AND interpretation ILIKE '%nyupptäckt%m%komponent%';
                 """).fetchnumpy()['id']

df = con.execute(""" 
                 SELECT id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()
negatives = set(negatives)
positives = set(positives)

df = df[df['id'].isin(negatives | positives)]

df = df[
    (df['fractions'].apply(len) == 6) &
    (df['boundaries'].apply(len) == 12)
]
df['label'] = df['id'].apply(
    lambda x: 1 if x in positives else 0
)


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.0).index


train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.retrain(cnn_train_dl,cnn_val_dl,model_path='../models/cnn_immunofixation.pth',patience=15)

Antal utan m-komponent i träningsdatan: 6295
Antal med m-komponent i träningsdatan: 2959
Epoch   0 | train: 1.2407 | val: 0.6071 | acc: 33.74% | AUC: 0.502
  -> ny bästa modell sparad till ../models/cnn_immunofixation.pth
Epoch   1 | train: 0.5980 | val: 0.5555 | acc: 36.42% | AUC: 0.693
  -> ny bästa modell sparad till ../models/cnn_immunofixation.pth
Epoch   2 | train: 0.5451 | val: 0.5168 | acc: 38.68% | AUC: 0.716
  -> ny bästa modell sparad till ../models/cnn_immunofixation.pth
Epoch   3 | train: 0.5262 | val: 0.4884 | acc: 38.07% | AUC: 0.749
  -> ny bästa modell sparad till ../models/cnn_immunofixation.pth
Epoch   4 | train: 0.5179 | val: 0.5231 | acc: 58.02% | AUC: 0.763
Epoch   5 | train: 0.5113 | val: 0.4865 | acc: 36.42% | AUC: 0.796
  -> ny bästa modell sparad till ../models/cnn_immunofixation.pth
Epoch   6 | train: 0.4952 | val: 0.4620 | acc: 45.68% | AUC: 0.798
  -> ny bästa modell sparad till ../models/cnn_immunofixation.pth
Epoch   7 | train: 0.4799 | val: 0.4610 | acc:

In [17]:
predict_fn = get_predict_fn(cnn_suffix = 'immunofixation',ae_suffix='first_time')
test_rows = test_rows[test_rows['label'].isin([0,1])]

result = predict_fn(test_rows)
evaluate(result,threshold=0.5,proportion=70)

              precision    recall  f1-score   support

     Negativ       0.93      0.78      0.85       745
     Positiv       0.66      0.88      0.75       351

    accuracy                           0.81      1096
   macro avg       0.80      0.83      0.80      1096
weighted avg       0.84      0.81      0.82      1096

[[584 161]
 [ 42 309]]
Accuracy:  81.48%
FN-rate:   11.97%  (farliga missade fall)
FP-rate:   21.61%  (onödiga larm)


(array([0.9937115 , 0.7772805 , 0.9694337 , ..., 0.939833  , 0.18907686,
        0.25031412], shape=(1096,), dtype=float32),
 array([1, 1, 1, ..., 1, 0, 0], shape=(1096,)),
 20        1
 108       1
 505       1
 790       1
 798       1
          ..
 172182    0
 172197    1
 172201    1
 172211    0
 172378    0
 Name: proportion_gamma_region, Length: 1096, dtype: int64,
 array([155130, 153295, 144120, ..., 176094, 177036, 148187], shape=(1096,)))